# Participant Pipeline



In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, Dataset, DataLoader


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
DATA_DIR = Path("competition_data")

def _shard_idx(p: Path):
    return int(p.stem.split("_")[-1])

x_candidates = sorted(DATA_DIR.glob("train_codeword_x_shard_*.csv"), key=_shard_idx)
y_candidates = sorted(DATA_DIR.glob("train_noisy_y_shard_*.csv"), key=_shard_idx)

x_map = {_shard_idx(p): p for p in x_candidates}
y_map = {_shard_idx(p): p for p in y_candidates}
common_shards = sorted(set(x_map).intersection(y_map))

train_x_files = [x_map[i] for i in common_shards]
train_y_files = [y_map[i] for i in common_shards]
test_y_path = DATA_DIR / "test_noisy_y_public.csv"

assert len(train_x_files) > 0
assert test_y_path.exists()

test_y_df = pd.read_csv(test_y_path)
print(f"train shards={len(train_x_files)}, test rows={len(test_y_df)}")


In [ ]:
class SingleShardByIdDataset(IterableDataset):
    def __init__(self, x_file, y_file, mode="train", val_mod=20, val_rem=0, chunksize=4096):
        self.x_file = x_file
        self.y_file = y_file
        self.mode = mode
        self.val_mod = val_mod
        self.val_rem = val_rem
        self.chunksize = chunksize

    def __iter__(self):
        x_iter = pd.read_csv(self.x_file, chunksize=self.chunksize)
        y_iter = pd.read_csv(self.y_file, chunksize=self.chunksize)
        for x_chunk, y_chunk in zip(x_iter, y_iter):
            ids = x_chunk["id"].to_numpy()
            is_val = (ids % self.val_mod) == self.val_rem
            use_mask = ~is_val if self.mode == "train" else is_val
            if use_mask.sum() == 0:
                continue

            x_np = x_chunk.loc[use_mask].drop(columns=["id"]).to_numpy(dtype=np.float32)
            y_np = y_chunk.loc[use_mask].drop(columns=["id"]).to_numpy(dtype=np.float32)

            for y_row, x_row in zip(y_np, x_np):
                yield torch.from_numpy(y_row), torch.from_numpy(x_row)

class PublicTestDataset(Dataset):
    def __init__(self, y_df):
        self.ids = y_df["id"].to_numpy(dtype=np.int64)
        self.y = y_df.drop(columns=["id"]).to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], torch.from_numpy(self.y[idx])


In [ ]:
import json

# Read code configuration from manifest and load the parity-check matrix H.
manifest_path = DATA_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    code_type = manifest.get("code_type", "POLAR")
    code_n = manifest.get("code_n", 64)
    code_k = manifest.get("code_k", 32)
else:
    code_type, code_n, code_k = "POLAR", 64, 32

class Code:
    pass

code = Code()
code.code_type = code_type
code.n = code_n
code.k = code_k
code.ecct_type = "NSM"


def Read_pc_matrixrix_alist(fileName):
    with open(fileName, "r", encoding="utf-8") as file:
        lines = file.readlines()
        columnNum, rowNum = np.fromstring(lines[0].rstrip("\n"), dtype=int, sep=" ")
        H = np.zeros((rowNum, columnNum)).astype(int)
        for column in range(4, 4 + columnNum):
            nonZeroEntries = np.fromstring(lines[column].rstrip("\n"), dtype=int, sep=" ")
            for row in nonZeroEntries:
                if row > 0:
                    H[row - 1, column - 4] = 1
        return H


def row_reduce(mat, ncols=None):
    assert mat.ndim == 2
    ncols = mat.shape[1] if ncols is None else ncols
    mat_row_reduced = mat.copy()
    p = 0
    for j in range(ncols):
        idxs = p + np.nonzero(mat_row_reduced[p:, j])[0]
        if idxs.size == 0:
            continue
        mat_row_reduced[[p, idxs[0]], :] = mat_row_reduced[[idxs[0], p], :]
        idxs = np.nonzero(mat_row_reduced[:, j])[0].tolist()
        idxs.remove(p)
        mat_row_reduced[idxs, :] = mat_row_reduced[idxs, :] ^ mat_row_reduced[p, :]
        p += 1
        if p == mat_row_reduced.shape[0]:
            break
    return mat_row_reduced, p


def get_generator(pc_matrix_):
    assert pc_matrix_.ndim == 2
    pc_matrix = pc_matrix_.copy().astype(bool).transpose()
    pc_matrix_I = np.concatenate((pc_matrix, np.eye(pc_matrix.shape[0], dtype=bool)), axis=-1)
    pc_matrix_I, p = row_reduce(pc_matrix_I, ncols=pc_matrix.shape[1])
    return row_reduce(pc_matrix_I[p:, pc_matrix.shape[1]:])[0]


def get_standard_form(pc_matrix_):
    pc_matrix = pc_matrix_.copy().astype(bool)
    next_col = min(pc_matrix.shape)
    for ii in range(min(pc_matrix.shape)):
        while True:
            rows_ones = ii + np.where(pc_matrix[ii:, ii])[0]
            if len(rows_ones) == 0:
                new_shift = np.arange(ii, min(pc_matrix.shape) - 1).tolist() + [min(pc_matrix.shape) - 1, next_col]
                old_shift = np.arange(ii + 1, min(pc_matrix.shape)).tolist() + [next_col, ii]
                pc_matrix[:, new_shift] = pc_matrix[:, old_shift]
                next_col += 1
            else:
                break
        pc_matrix[[ii, rows_ones[0]], :] = pc_matrix[[rows_ones[0], ii], :]
        other_rows = pc_matrix[:, ii].copy()
        other_rows[ii] = False
        pc_matrix[other_rows] = pc_matrix[other_rows] ^ pc_matrix[ii]
    return pc_matrix.astype(int)


def Get_Generator_and_Parity(code, standard_form=False):
    n, k = code.n, code.k
    path_pc_mat = Path("Codes_DB") / f"{code.code_type}_N{n}_K{k}"

    if code.code_type in ["POLAR", "BCH", "LTE_TURBO"]:
        ParityMatrix = np.loadtxt(str(path_pc_mat) + ".txt")
    elif code.code_type in ["CCSDS", "LDPC", "MACKAY"]:
        ParityMatrix = Read_pc_matrixrix_alist(str(path_pc_mat) + ".alist")
    else:
        raise ValueError(f"Wrong code type: {code.code_type}")

    ParityMatrix_Original = ParityMatrix.copy().astype(int)
    pc_matrix_I, p = row_reduce(ParityMatrix_Original.astype(bool), ncols=ParityMatrix_Original.shape[1])
    H_sys_dense = pc_matrix_I.astype(int)
    H_for_G = get_standard_form(H_sys_dense).astype(int)
    GeneratorMatrix = get_generator(H_for_G)
    return GeneratorMatrix.astype(float), ParityMatrix.astype(float)


_, H_np = Get_Generator_and_Parity(code)
H_tensor = torch.from_numpy(H_np.astype(np.int32)).to(device)

n = code_n
n_minus_k = n - code_k
print(f"Loaded {code_type} (N={n}, K={code_k})")
print(f"H shape = {H_tensor.shape}")


In [ ]:
import torch.nn.functional as F

class SeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()
        self.depthwise = nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size, groups=in_channels, padding=padding)
        self.pointwise = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        
    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class ResidualBlock1d(nn.Module):
    def __init__(self, filters):
        super().__init__()
        self.conv1 = SeparableConv1d(filters, filters, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(filters)
        self.conv2 = SeparableConv1d(filters, filters, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(filters)
        
    def forward(self, x_in):
        x = self.conv1(x_in)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        return x_in + x

class ResidualDecoder(nn.Module):
    def __init__(self, n, n_minus_k):
        super().__init__()
        self.n = n
        
        self.conv_feat = nn.Conv1d(2, 64, kernel_size=1, padding=0)
        
        self.syn_dense = nn.Linear(n_minus_k, n)
        self.conv_combined = nn.Conv1d(65, 64, kernel_size=1, padding=0)
        
        self.res_block = ResidualBlock1d(64)
        
        self.fc = nn.Sequential(
            nn.Linear(64 * n, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, n)
        )

    def forward(self, hd_bits, llr, syndrome):
        x_input = torch.stack([hd_bits, llr], dim=1) # (B, 2, n)
        x_feat = F.relu(self.conv_feat(x_input))
        
        x_syn = F.relu(self.syn_dense(syndrome)).unsqueeze(1)
        
        x_combined = torch.cat([x_feat, x_syn], dim=1) # (B, 65, n)
        x_combined = F.relu(self.conv_combined(x_combined))
        
        x_combined = self.res_block(x_combined)
        
        x_flat = x_combined.view(x_combined.size(0), -1)
        
        output = self.fc(x_flat)
        return output

model = ResidualDecoder(n, n_minus_k).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def process_batch(y, x):

    hd_bits = (y < 0).float()

    llr = y.float() 
    
    syndrome_float = torch.matmul(hd_bits, H_tensor.float().T) 
    syndrome = (syndrome_float.int() % 2).float()
    
    target_flip = (x.int() ^ hd_bits.int()).float()
    
    return hd_bits, llr, syndrome, target_flip

def ber_from_logits(logits, hd_bits, original_x):

    flip_pred = (torch.sigmoid(logits) > 0.5).int()
    predicted_x = hd_bits.int() ^ flip_pred

    return (predicted_x != original_x.int()).float().mean().item()

def train_one_epoch(train_loader):
    model.train()
    criterion.train()
    total_loss, total_ber, total_n = 0.0, 0.0, 0
    for y, x in train_loader:
        y, x = y.to(device), x.to(device)
        hd_bits, llr, syndrome, target_flip = process_batch(y, x)
        
        outputs = model(hd_bits, llr, syndrome)
        loss = criterion(outputs, target_flip)
        
        opt.zero_grad()
        loss.backward()
        
        opt.step()
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_ber += ber_from_logits(outputs.detach(), hd_bits, x) * bs
        total_n += bs
    return total_loss / max(total_n, 1), total_ber / max(total_n, 1)

@torch.no_grad()
def eval_one_epoch(val_loader):
    model.eval()
    criterion.eval()
    total_loss, total_ber, total_n = 0.0, 0.0, 0
    for y, x in val_loader:
        y, x = y.to(device), x.to(device)
        hd_bits, llr, syndrome, target_flip = process_batch(y, x)
        
        outputs = model(hd_bits, llr, syndrome)
        loss = criterion(outputs, target_flip)
        
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_ber += ber_from_logits(outputs, hd_bits, x) * bs
        total_n += bs
    return total_loss / max(total_n, 1), total_ber / max(total_n, 1)

In [ ]:
BATCH_SIZE = 512
EPOCHS = min(200, len(train_x_files))
VAL_MOD, VAL_REM = 20, 0
PATIENCE = 50

best_val = float("inf")
bad_epochs = 0

for ep in range(1, EPOCHS + 1):
    shard_idx = (ep - 1) % len(train_x_files)
    x_file, y_file = train_x_files[shard_idx], train_y_files[shard_idx]

    train_ds = SingleShardByIdDataset(x_file, y_file, mode="train", val_mod=VAL_MOD, val_rem=VAL_REM)
    val_ds = SingleShardByIdDataset(x_file, y_file, mode="val", val_mod=VAL_MOD, val_rem=VAL_REM)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    train_loss, train_ber = train_one_epoch(train_loader)
    val_loss, val_ber = eval_one_epoch(val_loader)

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0
        torch.save(model.state_dict(), "best_participant_residual_decoder.pt")
    else:
        bad_epochs += 1

    print(f"epoch={ep} shard={shard_idx:03d} train_loss={train_loss:.4e} train_BER={train_ber:.4e} val_loss={val_loss:.4e} val_BER={val_ber:.4e} bad_epochs={bad_epochs}")

    if bad_epochs >= PATIENCE:
        print("early stopping")
        break

In [ ]:
model.load_state_dict(torch.load("best_participant_residual_decoder.pt", map_location=device))
model.eval()

test_ds = PublicTestDataset(test_y_df)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)
rows = []
with torch.no_grad():
    for ids, y in test_loader:
        y = y.to(device)
        hd_bits = (y < 0).float()
        llr = y.float() 
        
        syndrome_float = torch.matmul(hd_bits, H_tensor.float().T) 
        syndrome = (syndrome_float.int() % 2).float()
        
        outputs = model(hd_bits, llr, syndrome)
        
        flip_pred = (torch.sigmoid(outputs) > 0.5).int()
        
        bits = (hd_bits.int() ^ flip_pred).cpu().numpy()
        
        for sid, b in zip(ids.numpy(), bits):
            rows.append([int(sid), *b.tolist()])

submission = pd.DataFrame(rows, columns=["id"] + [f"bit_{i}" for i in range(n)])
submission.to_csv("submission_residual_decoder.csv", index=False)
print(submission.shape)
submission.head()